# Rotten Fruit Detector — Colab Training (KerasHub RetinaNet)

TensorFlow-native object detection with **KerasHub RetinaNet** — no TF Object Detection API, no protobuf/protoc setup. `main()` reads the Roboflow CSV export into a tf.data pipeline, fine-tunes RetinaNet with `model.fit()`, evaluates, and saves the model.

Expected `dataset.zip` (in Drive) contents:

```text
train/_annotations.csv + *.jpg
valid/_annotations.csv + *.jpg
test/_annotations.csv  + *.jpg
```
(Roboflow **TensorFlow Object Detection** CSV export.) Split folders may sit at the zip root or one level deep — the notebook auto-detects.

Set **Runtime → Change runtime type → GPU** first. Smoke-test with `epochs=1` before a full run; lower `batch_size` (e.g. 2) if you hit CUDA OOM.

In [ ]:
# 1. Clone the repo and install deps (TensorFlow + KerasHub)
REPO_URL = "https://github.com/YOUR_USERNAME/rotten-fruit-detection-ai.git"

!git clone {REPO_URL} /content/rotten-fruit-detection-ai
%cd /content/rotten-fruit-detection-ai
!pip install -q -r requirements.txt

In [ ]:
# 2. Mount Drive, unzip the dataset, and locate the split folders
from google.colab import drive
import zipfile
from pathlib import Path

drive.mount("/content/drive")

EXTRACT_DIR = Path("dataset")
with zipfile.ZipFile("/content/drive/MyDrive/dataset.zip") as archive:
    archive.extractall(EXTRACT_DIR)

def find_dataset_root(base):
    base = Path(base)
    for path in [base, *(p for p in base.rglob("*") if p.is_dir())]:
        if (path / "train").is_dir():
            return path
    raise FileNotFoundError(f"No 'train' folder found under {base}")

DATA_DIR = find_dataset_root(EXTRACT_DIR)
print("Dataset root:", DATA_DIR)
print("Contains:", sorted(p.name for p in DATA_DIR.iterdir() if p.is_dir()))

In [ ]:
# 3. Train -> evaluate -> save (KerasHub uses the TensorFlow backend)
import os
os.environ["KERAS_BACKEND"] = "tensorflow"  # set BEFORE importing keras/keras_hub

import sys
sys.path.insert(0, "src")

from train import main

model = main(data_dir=DATA_DIR, epochs=20, batch_size=4, image_size=640)

In [ ]:
# 4. Copy the saved model back to Drive
!cp retinanet_fruit.keras /content/drive/MyDrive/